[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/stokes_swimmer/stokes_swimmer.ipynb)

# Low-Reynolds-Number Swimming and Regularized Stokeslets

In a slow, viscous flow where inertia is absent, a swimmer has no coasting: at every instant the forces its body exerts on the fluid and their moments sum to zero, a vanishing wrench (`Forque`). In the projective geometric algebra of the plane (`PGA2D`), a force concentrated in a blob of radius `BEAD_RADIUS` drives a fluid velocity (`Direction`) per unit force (`Force`) given by the regularized Stokeslet, a map `Direction <- Force` that reduces at zero separation to `(2 / BEAD_RADIUS) * Force.dual()`. Over a swimmer's bead points `sites` (`Point`), those pairwise maps form a field operator `M: Direction[beads] <- Force[beads]`. Leaving the body's rigid rate `Twist` open in the commutator with the beads, `rigid = sites.commutator(Twist).field(0)` (`Direction[beads] <- Twist`) gives the velocity of every bead per unit body twist, and its incidence adjugate `rigid.adjugate()` (`Forque <- Force[beads]`) sums both forces and torques across the beads into the net body wrench. A reciprocal flapper retraces a line in hinge-angle space and returns to its starting pose `Motor` after every cycle, Purcell's scallop theorem; Purcell's three-link swimmer traces a loop in hinge-angle space and walks steadily through the fluid.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import concatenate
from examples.animation import save_figure, show_animation
from examples.mechanics.stokes_swimmer import core, render
from examples.mechanics.stokes_swimmer.core import ORIGIN, flow_field, separation, three_link_shape

ga = core.ga
mv = core.mv
Point = ga.gatype.antivector()
Twist = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
Force = ga.gatype.vector().derive.nondegenerate
Direction = Force.dual().output_gatype

## 1. The regularized Stokeslet and the grand resistance map

For two points `target` and `source`, the Euclidean vector between them is read from their joining line at the origin, `r = (target & source) | ORIGIN` (`Force`). With `r2 = r.norm_squared() + BEAD_RADIUS**2`, the fluid velocity per unit blob force is `((r2 + BEAD_RADIUS**2) * Force.dual() + r.dual() * (r | Force)) / (r2 * r2.square_root())` (`Direction <- Force`). Evaluated between all pairs of beads `sites`, `.field(0, 1)` packages the pairwise blocks into the mobility operator `M: Direction[beads] <- Force[beads]`. Leaving the body's twist `Twist` open, `rigid = sites.commutator(Twist).field(0)` gives the velocity of every bead per unit body twist, and `rigid.adjugate()` sums the bead forces into a wrench in `Forque`. Composing `rigid.adjugate()` with `M.solve(rigid)` builds the swimmer's grand resistance map `Forque <- Twist`.

In the matrix notation of microhydrodynamics, the mobility operator reads as a $2N \times 2N$ block Oseen matrix, the rigid kinematics as a $2N \times 3$ matrix of identity and lever-arm blocks, and the grand resistance map as the $3 \times 3$ Schur-complement resistance matrix.

In [ ]:
HINGE_HALF_SPAN = 0.8
CENTRE_OFFSETS = np.array([-0.5, 0.0, 0.5])
ARM_OFFSETS = np.array([0.35, 0.7, 1.05])
BEAD_RADIUS = 0.14


def stokeslet(separations: Force, radius: float):
    """Fluid velocity (`Direction`) per unit blob force (`Force`) at `separations`."""
    r2 = separations.norm_squared() + radius**2                             # [...] Scalar
    return ((r2 + radius**2) * Force.dual() + separations.dual() * (separations | Force)) / (r2 * r2.square_root())


def swim(sites: Point, shape_velocity: Direction, radius: float):
    """The body-frame rigid rate (`Twist`) and bead forces (`Force[beads]`) of a force- and torque-free swimmer."""
    pairs = separation(sites[..., :, None], sites[..., None, :])            # [..., beads, beads] Force
    M = stokeslet(pairs, radius).field(0, 1)                                # [...] Direction[beads] <- Force[beads]
    rigid = sites.commutator(Twist).field(0)                                # [...] Direction[beads] <- Twist
    resistance = rigid.adjugate()(M.solve(rigid))                           # [...] Forque <- Twist
    body_rate = -resistance.solve(rigid.adjugate()(M.solve(shape_velocity.field())))   # [...] Twist
    forces = M.solve(rigid(body_rate) + shape_velocity.field())             # [...] Force[beads]
    return body_rate, forces

## 2. Purcell's stroke beside a reciprocal stroke

Over three cycles, the first swimmer moves its two hinges ninety degrees out of phase around a circle in hinge-angle space, while the second moves both hinges in phase along an asymmetric line. Each step multiplies `(body_rate * (-dt / 2)).exp()` on the right of the swimmer's pose `Motor`, chained across all steps by `steps.reverse().cumprod(axis=0).reverse()`. On the left are Purcell's swimmer, the green forces its nine beads push on the fluid with, the grey Stokes flow they drive, and the red loop its centre traces; in the middle are the two swimmers' horizontal displacements over the three cycles, where the dashed orange reciprocal swimmer lunges forward and back each cycle to return to zero while the solid blue Purcell swimmer steps steadily across the plane; on the right are the circle and the line in hinge-angle space.

In [ ]:
AMPLITUDE = 1.05
CYCLES = 3
STEPS_PER_CYCLE = 60
EVERY = 3
GRID_X = np.linspace(-2.8, 2.8, 25)
GRID_Y = np.linspace(-1.9, 1.9, 17)

steps = CYCLES * STEPS_PER_CYCLE
dt = 2 * np.pi / STEPS_PER_CYCLE
phases = np.arange(steps) * dt                                              # [steps]

angles = np.stack([
    np.stack([AMPLITUDE * np.cos(phases), AMPLITUDE * np.sin(phases)], axis=-1),
    np.stack([AMPLITUDE * np.cos(phases), -0.5 * AMPLITUDE * np.cos(phases)], axis=-1),
], axis=1)                                                                  # [steps, swimmers, 2]
rates = np.stack([
    np.stack([-AMPLITUDE * np.sin(phases), AMPLITUDE * np.cos(phases)], axis=-1),
    np.stack([-AMPLITUDE * np.sin(phases), 0.5 * AMPLITUDE * np.sin(phases)], axis=-1),
], axis=1)                                                                  # [steps, swimmers, 2]

sites, shape_velocity = three_link_shape(HINGE_HALF_SPAN, CENTRE_OFFSETS, ARM_OFFSETS, angles, rates)
body_rate, forces = swim(sites, shape_velocity, BEAD_RADIUS)                # [steps, swimmers] Twist, Force[beads]

start_motor = (mv.xy * np.array([0.0, 0.0])).exp()                          # [swimmers] Motor
step_motors = (body_rate * (-dt / 2.0)).exp()                               # [steps, swimmers] Motor
motors = concatenate([start_motor[None], start_motor * step_motors.reverse().cumprod(axis=0).reverse()], axis=0)
positions = motors >> ORIGIN                                                # [steps + 1, swimmers] Point
world_sites = motors[:-1, :, None] >> sites                                 # [steps, swimmers, beads] Point
world_forces = (motors[:-1, :, None] >> forces.batch().dual()).dual()       # [steps, swimmers, beads] Force

gx, gy = np.meshgrid(GRID_X, GRID_Y, indexing="xy")
grid = mv.point(np.stack([gx, gy], axis=-1))                                # [grid_y, grid_x] Point
frame_slice = slice(0, steps, EVERY)
flow = flow_field(
    grid[None],
    world_sites[frame_slice, 0, None, None],
    world_forces[frame_slice, 0, None, None],
    BEAD_RADIUS,
)                                                                           # [frames, grid_y, grid_x] Direction

save_figure(render.draw_summary(angles, positions, world_sites, world_forces, grid, flow[0]), "stokes_swimmer");

In [ ]:
show_animation(render.animate_swimmers(positions, world_sites, grid, flow, EVERY), "stokes_swimmer", 60)

In [ ]:
# checks
self_mobility = stokeslet(0.0 * mv.x + 0.0 * mv.y, BEAD_RADIUS)
np.testing.assert_allclose((self_mobility - (2.0 / BEAD_RADIUS) * Force.dual()).kernel, 0.0, atol=1e-12)

rigid = sites.commutator(Twist).field(0)
np.testing.assert_allclose(rigid.adjugate()(forces).kernel, 0.0, atol=1e-12)

cycle_motors = motors[STEPS_PER_CYCLE::STEPS_PER_CYCLE]                     # [cycles, swimmers] Motor
np.testing.assert_allclose((cycle_motors[:, 1] - start_motor[1]).kernel, 0.0, atol=1e-4)
cycle_centres = positions[STEPS_PER_CYCLE::STEPS_PER_CYCLE]                 # [cycles, swimmers] Point
purcell_x = (separation(cycle_centres[:, 0], ORIGIN) | mv.x).to_array()
step_advances = np.diff(np.concatenate([[0.0], purcell_x]))
assert np.all(np.abs(step_advances) > 0.04)
np.testing.assert_allclose(step_advances, step_advances[0], rtol=1e-2)